# PXD024871 — Colab compute runner

**This notebook orchestrates; it does not analyse.** Every scientific step lives
in `scripts/` in the repository, so the run records, checksums and environment
capture that `PROJECT_PROMPT.md` requires keep working. If you find yourself
writing analysis in a cell, it belongs in a script instead — notebook cells
execute out of order and leave no reproducible trace.

## Which runtime to pick

**Use a CPU runtime for extraction.** Downloading and extracting the containers
is network- and SQLite-bound; a GPU sits idle through all of it and you would
burn compute units for nothing.

Runtime → Change runtime type → **CPU** for this notebook's extraction job.
Switch to **T4 GPU** later, for training only.

**T4 is the chosen training accelerator.** Before committing to a full grid,
measure one epoch and compare against the CPU baseline in `RUN_LOG.md` run-002
(122,648 peptides/sec). The §11 architecture is small, so it may be
kernel-launch-bound rather than compute-bound — a 10× speedup is a safer
expectation than 50×. If one epoch on T4 does not beat the CPU baseline by a
wide margin, say so before spending the grid: the honest finding would be that
this model is too small to use a GPU well.

## What this does

1. Clone the repo at the working branch
2. Install pinned dependencies
3. Capture this machine as a new `env_id`
4. Run the resumable batch extraction (48 containers, ~46 GiB)
5. Commit and push the extracts back

Peak disk is one container plus its extract (~12 GiB worst case), never the
whole set.

## Before you start

Add a GitHub token with `repo` scope to Colab Secrets (key icon, left sidebar)
named **`GH_TOKEN`**. The token is read from Secrets, never printed, and is
passed to git through a transient credential helper so it does not land in
`.git/config`.

In [ ]:
# 1. What machine did we get?
import os, shutil, subprocess
print("Colab release :", os.environ.get("COLAB_RELEASE_TAG", "not Colab"))
print("CPU count     :", os.cpu_count())
print("Disk free     : %.1f GiB" % (shutil.disk_usage("/content").free / 2**30))
print()
if shutil.which("nvidia-smi"):
    print(subprocess.run(["nvidia-smi","--query-gpu=name,memory.total",
                          "--format=csv,noheader"], capture_output=True, text=True).stdout.strip())
    print("\nNOTE: extraction does not use the GPU. If you are only extracting,")
    print("switch to a CPU runtime and save these compute units for training.")
else:
    print("No GPU — correct runtime for the extraction job.")

In [ ]:
# 2. Clone the repo. Token is read from Secrets and never echoed.
from google.colab import userdata
import os, subprocess, pathlib

GH_TOKEN = userdata.get("GH_TOKEN")          # add this in Colab Secrets first
REPO     = "hafijbd/PXD024871_mapping"
BRANCH   = "claude/mapping-research-project-prompt-glsess"
WORK     = pathlib.Path("/content/PXD024871_mapping")

os.environ["GH_TOKEN"] = GH_TOKEN            # consumed by the helper in step 6
# Token kept out of .git/config: supplied per-command via a credential helper.
HELPER = ('!f() { echo username=x-access-token; echo "password=$GH_TOKEN"; }; f')

if not WORK.exists():
    subprocess.run(["git", "-c", f"credential.helper={HELPER}", "clone",
                    "--branch", BRANCH, f"https://github.com/{REPO}.git", str(WORK)],
                   check=True)
os.chdir(WORK)
subprocess.run(["git", "config", "user.name",  "Claude Code (Colab runner)"], check=True)
subprocess.run(["git", "config", "user.email", "noreply@anthropic.com"], check=True)
print(subprocess.run(["git","log","--oneline","-1"], capture_output=True, text=True).stdout)

In [ ]:
# 3. Pinned dependencies. Phase A needs only the stdlib; this is for the
#    analysis stages and keeps the environment record honest.
!pip install -q -r requirements.txt
!python -c "import numpy, scipy; print('numpy', numpy.__version__, '| scipy', scipy.__version__)" 

In [ ]:
# 4. Record this machine. Pick the next free env id (env-001 is the cloud
#    session that did Phase A).
ENV_ID = "env-002"

out = subprocess.run(["python3","-I","scripts/capture_environment.py", ENV_ID],
                     capture_output=True, text=True).stdout
print(out)
with open("ENVIRONMENT.md","a") as fh:
    fh.write(out)
print(f"appended {ENV_ID} to ENVIRONMENT.md — cite it in the run record")

In [ ]:
# 5. Extract. RESUMABLE: if the session dies, just re-run this cell.
#    Completed units are skipped; a container left behind by a killed run is
#    reconciled and checksum-verified before anything reads it.
#
#    --limit lets you do it in chunks that fit inside one Colab session.
#    Drop --limit to attempt all remaining in one go.

!python3 -I scripts/batch_extract.py --limit 12

In [ ]:
# 6. Push the extracts back. Re-run after each extraction chunk so progress
#    survives the session ending.
import subprocess
HELPER = ('!f() { echo username=x-access-token; echo "password=$GH_TOKEN"; }; f')

subprocess.run(["git","add","data/derived/peptides","data/raw/S3/provenance.jsonl",
                "ENVIRONMENT.md"], check=True)
st = subprocess.run(["git","status","--short"], capture_output=True, text=True).stdout
print(st or "(nothing to commit)")

if st.strip():
    n = len(list(pathlib.Path("data/derived/peptides").glob("*.csv*")))
    subprocess.run(["git","commit","-m",
        f"Extract peptide tables on {ENV_ID}: {n} of 52 units complete\n\n"
        "Produced by scripts/batch_extract.py on a Colab runtime. Each container\n"
        "was checksum-verified against the publisher before extraction and\n"
        "deleted afterwards; provenance records are in data/raw/S3.\n\n"
        "Co-Authored-By: Claude Opus 5.5 <noreply@anthropic.com>"], check=True)
    subprocess.run(["git","-c",f"credential.helper={HELPER}","push","origin",BRANCH], check=True)
    print("pushed")

## If the session dies

Re-run cells 2 → 6. Nothing is lost that was pushed, and nothing is redone that
was finished — `batch_extract.py` skips units whose extract already exists and
reconciles any half-transferred container before reading it.

## Progress check

```python
!ls data/derived/peptides/*.csv* | wc -l   # units complete, of 52
!python3 -I scripts/batch_extract.py --dry-run | head -3
```

## Afterwards

Once all 52 units are extracted, the open work is **G4** — negative-control
construction and the frozen peptide table — which needs two decisions closed
first (`DECISION_LOG.md` D002 for the negative strategy and class ratio, D004
for the confidence threshold).

Training is the step that wants the **T4**. Switch the runtime then, not now,
and record it as a separate `env_id` (`env-003`): a run on different hardware is
a different run, and `SECTIONS.md` §14 says so explicitly. Re-run
`scripts/benchmark_compute.py` on the T4 before training — §14's limits were
derived from CPU throughput and do not port.

## What this notebook deliberately does not do

It does not fit models, compute metrics, choose thresholds, or construct
negatives. Those are preregistration-gated decisions, and running them from a
notebook cell would put untraceable analysis in front of the gates that exist to
prevent exactly that.